In [ ]:
import re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.metrics import accuracy_score, f1_score, classification_report

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier

## 1. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/aurora-gate-expense-categorization-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/aurora-gate-expense-categorization-challenge/test.csv')
all_df = pd.concat([train, test], ignore_index=True)

print(f"Train: {train.shape}, Test: {test.shape}")
train['category'].value_counts()


## 2. Feature Engineering

### 2a. Merchant Rules & Root Extraction

In [ ]:
state_pattern = r'\b(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VT|VA|WA|WV|WI|WY|DC)\b'

MERCHANT_RULES = [
    ('uber eats', 'Food & Dining'), ('doordash', 'Food & Dining'), ('grubhub', 'Food & Dining'),
    ('seamless', 'Food & Dining'), ('postmates', 'Food & Dining'), ('mcdonald', 'Food & Dining'),
    ('starbucks', 'Food & Dining'), ('chipotle', 'Food & Dining'), ('subway', 'Food & Dining'),
    ('burger king', 'Food & Dining'), ('domino', 'Food & Dining'), ('taco bell', 'Food & Dining'),
    ('dunkin', 'Food & Dining'), ('chick-fil-a', 'Food & Dining'), ('wendy', 'Food & Dining'),
    ('panera', 'Food & Dining'), ('pizza', 'Food & Dining'), ('cafe', 'Food & Dining'),
    ('bistro', 'Food & Dining'), ('restaurant', 'Food & Dining'), ('grill', 'Food & Dining'),
    ('lyft', 'Transportation'), ('uber', 'Transportation'), ('chevron', 'Transportation'),
    ('shell', 'Transportation'), ('bp', 'Transportation'), ('exxon', 'Transportation'),
    ('mobil', 'Transportation'), ('speedway', 'Transportation'), ('7-eleven gas', 'Transportation'),
    ('parking', 'Transportation'), ('metro', 'Transportation'), ('transit', 'Transportation'),
    ('toll', 'Transportation'), ('subway transit', 'Transportation'), ('airline', 'Transportation'),
    ('netflix', 'Subscriptions'), ('spotify', 'Subscriptions'), ('hulu', 'Subscriptions'),
    ('disney', 'Subscriptions'), ('hbo', 'Subscriptions'), ('apple.com/bill', 'Subscriptions'),
    ('youtube', 'Subscriptions'), ('amazon prime', 'Subscriptions'), ('nytimes', 'Subscriptions'),
    ('patreon', 'Subscriptions'), ('wsj', 'Subscriptions'), ('medium', 'Subscriptions'),
    ('walmart grocery', 'Groceries'), ('safeway', 'Groceries'), ('costco', 'Groceries'),
    ('whole foods', 'Groceries'), ('trader joe', 'Groceries'), ('aldi', 'Groceries'),
    ('kroger', 'Groceries'), ('target t-', 'Groceries'), ('publix', 'Groceries'),
    ('h-e-b', 'Groceries'), ('food lion', 'Groceries'), ('sprouts', 'Groceries'),
    ('cvs', 'Health & Fitness'), ('walgreens', 'Health & Fitness'), ('pharmacy', 'Health & Fitness'),
    ('planet fitness', 'Health & Fitness'), ('equinox', 'Health & Fitness'), ('gym', 'Health & Fitness'),
    ('clinic', 'Health & Fitness'), ('quest diag', 'Health & Fitness'), ('gaspari', 'Health & Fitness'),
    ('verizon', 'Bills & Utilities'), ('at&t', 'Bills & Utilities'), ('t-mobile', 'Bills & Utilities'),
    ('coned', 'Bills & Utilities'), ('electric', 'Bills & Utilities'), ('water dept', 'Bills & Utilities'),
    ('comcast', 'Bills & Utilities'), ('xfinity', 'Bills & Utilities'), ('geico', 'Bills & Utilities'),
    ('amazon', 'Shopping'), ('best buy', 'Shopping'), ('target.com', 'Shopping'),
    ('ebay', 'Shopping'), ('etsy', 'Shopping'), ('home depot', 'Shopping'), ('lowe', 'Shopping'),
    ('amc', 'Entertainment'), ('regal', 'Entertainment'), ('steam', 'Entertainment'),
    ('playstation', 'Entertainment'), ('nintendo', 'Entertainment'), ('xbox', 'Entertainment'),
    ('hotel', 'Travel'), ('airbnb', 'Travel'), ('expedia', 'Travel'), ('booking.com', 'Travel'),
    ('marriott', 'Travel'), ('hilton', 'Travel'), ('delta', 'Travel'), ('united air', 'Travel'),
]

RULE_CATS = sorted(set(c for _, c in MERCHANT_RULES))
RULE_IDX  = {c: i + 1 for i, c in enumerate(RULE_CATS)}

def get_rule_feature(desc):
    s = str(desc).lower()
    for kw, cat in MERCHANT_RULES:
        if kw in s:
            return RULE_IDX[cat]
    return 0

def extract_merchant_root(desc):
    if not isinstance(desc, str): return ''
    s = desc.upper()
    s = re.sub(state_pattern + r'$', '', s).strip()
    s = re.sub(r'#?\d+', '', s).strip()
    return re.sub(r'\s+', ' ', s).strip()

all_df['merchant_root'] = all_df['description'].apply(extract_merchant_root)
merchant_freq_map = all_df['merchant_root'].value_counts().to_dict()
print(f"Unique merchant roots: {len(merchant_freq_map)}")


### 2b. Tabular Feature Builder

In [ ]:
def build_features(df):
    feats = pd.DataFrame(index=df.index)
    desc = df['description'].astype(str)

    # Financial & Cents Patterns
    feats['amount']         = df['amount']
    feats['log_amount']     = np.log1p(df['amount'])
    cents = (df['amount'] * 100 % 100).round().astype(int)
    feats['cents']          = cents
    feats['is_round_dollar']= (cents == 0).astype(int)
    feats['is_99_cents']    = (cents == 99).astype(int)
    feats['is_95_cents']    = (cents == 95).astype(int)
    feats['is_50_cents']    = (cents == 50).astype(int)
    feats['amount_mod_5']   = ((cents == 0) & (df['amount'].astype(int) % 5 == 0)).astype(int)
    feats['amount_mod_10']  = ((cents == 0) & (df['amount'].astype(int) % 10 == 0)).astype(int)
    feats['amount_bin']     = pd.qcut(all_df['amount'], q=10, labels=False, duplicates='drop').loc[df.index]

    # Domain Rule Feature
    feats['rule_feat'] = desc.apply(get_rule_feature)

    # Text Structural Signals
    feats['char_len']        = desc.apply(len)
    feats['word_len']        = desc.apply(lambda x: len(x.split()))
    feats['digit_count']     = desc.apply(lambda x: sum(c.isdigit() for c in x))
    feats['digit_ratio']     = feats['digit_count'] / (feats['char_len'] + 1e-5)
    feats['uppercase_count'] = desc.apply(lambda x: sum(c.isupper() for c in x))
    feats['uppercase_ratio'] = feats['uppercase_count'] / (feats['char_len'] + 1e-5)
    feats['has_hash']        = desc.str.contains('#', regex=False).astype(int)
    feats['has_star']        = desc.str.contains(r'\*', regex=True).astype(int)
    feats['has_slash']       = desc.str.contains('/', regex=False).astype(int)
    feats['has_dot_com']     = desc.str.lower().str.contains(r'\.com', regex=True).astype(int)
    feats['has_store_num']   = desc.str.contains(r'#?\d{3,}', regex=True).astype(int)
    feats['has_state_code']  = desc.str.contains(state_pattern, regex=True).astype(int)

    # Merchant Frequency
    feats['merchant_freq'] = desc.apply(extract_merchant_root).map(merchant_freq_map).fillna(0)

    # Temporal Features
    dt = pd.to_datetime(df['date'])
    feats['year']           = dt.dt.year
    feats['month']          = dt.dt.month
    feats['day']            = dt.dt.day
    feats['dayofweek']      = dt.dt.dayofweek
    feats['is_weekend']     = (feats['dayofweek'] >= 5).astype(int)
    feats['is_month_start'] = dt.dt.is_month_start.astype(int)
    feats['is_month_end']   = dt.dt.is_month_end.astype(int)
    feats['is_payday']      = feats['day'].isin([1, 2, 14, 15, 16, 28, 29, 30, 31]).astype(int)
    feats['week_of_month']  = (feats['day'] - 1) // 7 + 1
    feats['month_sin'] = np.sin(2 * np.pi * feats['month'] / 12)
    feats['month_cos'] = np.cos(2 * np.pi * feats['month'] / 12)
    feats['day_sin']   = np.sin(2 * np.pi * feats['day'] / 31)
    feats['day_cos']   = np.cos(2 * np.pi * feats['day'] / 31)
    feats['dow_sin']   = np.sin(2 * np.pi * feats['dayofweek'] / 7)
    feats['dow_cos']   = np.cos(2 * np.pi * feats['dayofweek'] / 7)

    date_counts = all_df['date'].value_counts()
    feats['date_trans_count'] = df['date'].map(date_counts).fillna(0)

    return feats

train_feats = build_features(train)
print(f"Tabular feature shape: {train_feats.shape}")
train_feats.head()


### 2c. OOF Target Encoding for Merchant Root

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(train['category'])
num_classes = len(le.classes_)
print(f"Classes: {le.classes_.tolist()}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_te_feats = np.zeros((len(train), num_classes))
train['m_root'] = train['description'].apply(extract_merchant_root)

global_priors = pd.Series(y_train).value_counts(normalize=True).sort_index().values

for trn_idx, val_idx in skf.split(train, y_train):
    tr_df  = train.iloc[trn_idx]
    val_df = train.iloc[val_idx]
    tr_y   = y_train[trn_idx]

    counts  = pd.crosstab(tr_df['m_root'], tr_y)
    m_totals = counts.sum(axis=1)
    smooth_weight = 10.0
    smoothed_probs = (counts + smooth_weight * global_priors) / (m_totals.values[:, None] + smooth_weight)

    val_encoded = np.tile(global_priors, (len(val_df), 1))
    for i, root in enumerate(val_df['m_root']):
        if root in smoothed_probs.index:
            val_encoded[i] = smoothed_probs.loc[root].values
    oof_te_feats[val_idx] = val_encoded

print(f"OOF Target Encoding shape: {oof_te_feats.shape}")


## 3. Text Vectorization & Hybrid Feature Matrix

In [ ]:
def clean_text(s):
    if not isinstance(s, str): return ''
    s = s.lower()
    s = re.sub(r'[#*\-_/\\|@&%$]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

train_desc_clean = train['description'].apply(clean_text)

word_vec = TfidfVectorizer(ngram_range=(1, 3), max_features=2500, min_df=2, sublinear_tf=True)
X_word = word_vec.fit_transform(train_desc_clean)

char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=3500, min_df=2, sublinear_tf=True)
X_char = char_vec.fit_transform(train_desc_clean)

num_cols = list(train_feats.columns)
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(train_feats[num_cols].fillna(0).values)

X_all = hstack([X_num_scaled, oof_te_feats, X_word, X_char]).tocsr()
print(f"Hybrid Matrix Shape (TF-IDF based): {X_all.shape}")


## 4. Sentence Transformer Embeddings (GPU)
Extracts 384-dim dense embeddings from transaction text using `all-MiniLM-L6-v2` on RTX 3060.

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")


from huggingface_hub import login
login(token=hf_token)

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    from sentence_transformers import SentenceTransformer

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

    st_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    formatted_texts = [
        f"Transaction: {d} | Amount: ${a:.2f}"
        for d, a in zip(train['description'], train['amount'])
    ]
    st_embeddings = st_model.encode(formatted_texts, batch_size=64, show_progress_bar=True)
    print(f"SentenceTransformer Embeddings Shape: {st_embeddings.shape}")
    HAS_PYTORCH = True
except Exception as e:
    print(f"PyTorch / SentenceTransformer not available: {e}")
    HAS_PYTORCH = False


## 5. Model Definitions

In [ ]:
THRESHOLD = 0.92  # Macro F1 selection threshold

models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=250, learning_rate=0.04, num_leaves=31, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, class_weight='balanced',
        random_state=42, n_jobs=4, verbose=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=250, learning_rate=0.04, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=4, eval_metric='mlogloss'
    ),
    'CatBoost': cb.CatBoostClassifier(
        iterations=200, learning_rate=0.05, depth=6,
        auto_class_weights='Balanced', verbose=0, thread_count=4, random_seed=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=20, class_weight='balanced',
        random_state=42, n_jobs=4
    ),
    'LogisticRegression': LogisticRegression(
        max_iter=400, C=3.0, class_weight='balanced', random_state=42, n_jobs=4
    ),
    'LinearSVC': CalibratedClassifierCV(
        LinearSVC(C=1.5, class_weight='balanced', random_state=42, max_iter=2500), cv=3
    ),
    'MLP': MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=200,
        early_stopping=True, n_iter_no_change=5, random_state=42
    ),
}

print(f"Models to evaluate: {list(models.keys())}")
print(f"Selection threshold: Macro F1 >= {THRESHOLD}")


## 6. 5-Fold Stratified CV — TF-IDF Based Models

In [ ]:
results = []
oof_predictions = {}

for name, model in models.items():
    print(f"\n---> {name} ...", flush=True)
    oof_preds = np.zeros(len(train))
    oof_probs = np.zeros((len(train), num_classes))

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_all, y_train)):
        model.fit(X_all[trn_idx], y_train[trn_idx])

        if hasattr(model, 'predict_proba'):
            val_p = model.predict_proba(X_all[val_idx])
            oof_probs[val_idx] = val_p
            oof_preds[val_idx] = np.argmax(val_p, axis=1)
        else:
            p = model.predict(X_all[val_idx])
            oof_preds[val_idx] = p
            for i, cls in enumerate(p):
                oof_probs[val_idx[i], int(cls)] = 1.0

    acc        = accuracy_score(y_train, oof_preds)
    f1_macro   = f1_score(y_train, oof_preds, average='macro')
    f1_weighted = f1_score(y_train, oof_preds, average='weighted')

    oof_predictions[name] = {'preds': oof_preds, 'probs': oof_probs}
    results.append({'Model': name, 'CV Accuracy': acc, 'Macro F1': f1_macro, 'Weighted F1': f1_weighted})
    print(f"  {name:<20} | Acc: {acc:.4f} | Macro F1: {f1_macro:.4f}")


## 7. PyTorch Deep-Wide Neural Network (ST + Tabular Features)

In [ ]:
if HAS_PYTORCH:
    print("---> PyTorch Deep-Wide Net ...", flush=True)

    # Combine: tabular + OOF Target Encoding + SentenceTransformer embeddings
    X_pytorch = np.hstack([X_num_scaled, oof_te_feats, st_embeddings])

    pt_oof_preds = np.zeros(len(train))
    pt_oof_probs = np.zeros((len(train), num_classes))

    class DeepWideNet(nn.Module):
        def __init__(self, in_dim, out_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 256),
                nn.BatchNorm1d(256),
                nn.SiLU(),
                nn.Dropout(0.3),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.SiLU(),
                nn.Dropout(0.2),
                nn.Linear(128, out_dim),
            )
        def forward(self, x):
            return self.net(x)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_pytorch, y_train)):
        X_tr_t  = torch.FloatTensor(X_pytorch[trn_idx]).to(device)
        y_tr_t  = torch.LongTensor(y_train[trn_idx]).to(device)
        X_val_t = torch.FloatTensor(X_pytorch[val_idx]).to(device)

        loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)
        pt_model   = DeepWideNet(X_pytorch.shape[1], num_classes).to(device)
        criterion  = nn.CrossEntropyLoss()
        optimizer  = optim.AdamW(pt_model.parameters(), lr=1e-3, weight_decay=1e-4)

        pt_model.train()
        for _ in range(25):
            for bx, by in loader:
                optimizer.zero_grad()
                loss = criterion(pt_model(bx), by)
                loss.backward()
                optimizer.step()

        pt_model.eval()
        with torch.no_grad():
            val_probs = torch.softmax(pt_model(X_val_t), dim=1).cpu().numpy()

        pt_oof_probs[val_idx] = val_probs
        pt_oof_preds[val_idx] = np.argmax(val_probs, axis=1)
        print(f"  Fold {fold+1} done", flush=True)

    pt_acc = accuracy_score(y_train, pt_oof_preds)
    pt_f1  = f1_score(y_train, pt_oof_preds, average='macro')
    pt_w1  = f1_score(y_train, pt_oof_preds, average='weighted')

    oof_predictions['PyTorch_DeepWide'] = {'preds': pt_oof_preds, 'probs': pt_oof_probs}
    results.append({'Model': 'PyTorch_DeepWide', 'CV Accuracy': pt_acc, 'Macro F1': pt_f1, 'Weighted F1': pt_w1})
    print(f"  PyTorch_DeepWide      | Acc: {pt_acc:.4f} | Macro F1: {pt_f1:.4f}")
else:
    print("Skipped (PyTorch not available)")


## 8. Model Comparison Ranking

In [ ]:
res_df = pd.DataFrame(results).sort_values('Macro F1', ascending=False).reset_index(drop=True)
res_df.style.background_gradient(subset=['Macro F1', 'CV Accuracy'], cmap='YlGn')


## 9. Threshold Filtering (Macro F1 >= {THRESHOLD})

In [ ]:
qualifying_models = res_df[res_df['Macro F1'] >= THRESHOLD]

if len(qualifying_models) > 0:
    print(f"✅ {len(qualifying_models)} model(s) passed Macro F1 >= {THRESHOLD}:")
    selected_model_names = qualifying_models['Model'].tolist()
else:
    print(f"⚠️ No model reached Macro F1 >= {THRESHOLD}. Falling back to top 3.")
    selected_model_names = res_df.head(3)['Model'].tolist()

print(f"Selected: {selected_model_names}")
qualifying_models if len(qualifying_models) > 0 else res_df.head(3)


## 10. Blended Ensemble Evaluation

In [ ]:
blend_probs = np.zeros((len(train), num_classes))
for m_name in selected_model_names:
    blend_probs += oof_predictions[m_name]['probs'] / len(selected_model_names)

blend_preds     = np.argmax(blend_probs, axis=1)
blend_acc       = accuracy_score(y_train, blend_preds)
blend_f1        = f1_score(y_train, blend_preds, average='macro')
blend_weighted  = f1_score(y_train, blend_preds, average='weighted')

print(f"🎉 Blended Ensemble ({len(selected_model_names)} models):")
print(f"   CV Accuracy  : {blend_acc:.4f}")
print(f"   Macro F1     : {blend_f1:.4f}")
print(f"   Weighted F1  : {blend_weighted:.4f}")

print()
print(classification_report(y_train, blend_preds, target_names=le.classes_))
